# 跨境电商销量预测 - Chronos-2 SageMaker 部署

基于 Amazon Chronos-2 预训练时序模型，支持协变量（节假日、促销、广告等）

## 1. 安装依赖

In [ ]:
!pip install -U -q "sagemaker<3" pandas matplotlib

In [ ]:
import sys
sys.path.append("code_preprocess")

import json
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sagemaker import Session
from preprocess import preprocess_data, PAST_COVARIATES, FUTURE_COVARIATES

## 2. 配置参数

In [ ]:
# SageMaker 配置
ROLE = None  # SageMaker Notebook 中可设为 None
session = Session()
BUCKET = session.default_bucket()  # 或指定你的 S3 bucket
S3_PREFIX = "ecommerce-forecast"

# 预测参数
PREDICTION_LENGTH = 28
FREQ = "D"
MARKETPLACE = "US"

## 3. 上传数据到 S3

In [ ]:
# 上传本地数据到 S3
s3 = boto3.client("s3")

# 上传销量历史数据
s3.upload_file("data/sales_history.csv", BUCKET, f"{S3_PREFIX}/data/sales_history.csv")
s3.upload_file("data/sku_metadata.csv", BUCKET, f"{S3_PREFIX}/data/sku_metadata.csv")

print(f"数据已上传到 s3://{BUCKET}/{S3_PREFIX}/data/")

## 4. 从 S3 加载数据

In [ ]:
def load_data_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """从 S3 加载 CSV 数据"""
    s3_path = f"s3://{bucket}/{key}"
    return pd.read_csv(s3_path, parse_dates=["date", "launch_date"])

# 加载数据
raw_df = load_data_from_s3(BUCKET, f"{S3_PREFIX}/data/sales_history.csv")
sku_meta = pd.read_csv(f"s3://{BUCKET}/{S3_PREFIX}/data/sku_metadata.csv")

print(f"数据量: {len(raw_df):,} 行")
print(f"SKU数量: {raw_df['asin'].nunique()}")
print(f"日期范围: {raw_df['date'].min()} ~ {raw_df['date'].max()}")
raw_df.head()

In [ ]:
# 数据预处理
df = preprocess_data(raw_df, marketplace=MARKETPLACE)
print(f"特征数量: {len(df.columns)}")
df.head()

## 5. 数据探索

In [ ]:
# 月度销量趋势
monthly_sales = df.groupby(df["date"].dt.to_period("M"))["sales_quantity"].sum()

plt.figure(figsize=(14, 5))
monthly_sales.plot(kind="bar", color="#2E86AB")
plt.title("月度销量趋势")
plt.xlabel("月份")
plt.ylabel("销量")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 类目销量分布
category_sales = df.groupby("category")["sales_quantity"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
category_sales.plot(kind="bar", color="#E94F37")
plt.title("类目销量分布")
plt.xlabel("类目")
plt.ylabel("总销量")
plt.tight_layout()
plt.show()

## 6. 部署 Chronos-2 端点

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

js_model = JumpStartModel(
    model_id="pytorch-forecasting-chronos-2",
    instance_type="ml.g5.xlarge",  # GPU，可改为 ml.c5.xlarge (CPU)
    role=ROLE,
)
predictor = js_model.deploy()
print("端点部署完成")

## 7. 构建预测请求

In [ ]:
def build_chronos_payload(df, target_col="sales_quantity", id_col="asin", 
                          date_col="date", prediction_length=28, freq="D",
                          past_cov_cols=None, future_cov_cols=None,
                          marketplace="US"):
    """构建 Chronos-2 API 请求"""
    inputs = []
    
    for item_id, group in df.sort_values([id_col, date_col]).groupby(id_col):
        # 只预测活跃产品 (最近30天有销量)
        if group[date_col].max() < df[date_col].max() - timedelta(days=30):
            continue
        
        entry = {
            "target": group[target_col].tolist(),
            "item_id": str(item_id),
            "start": group[date_col].iloc[0].isoformat(),
        }
        
        # 历史协变量
        if past_cov_cols:
            valid_cols = [c for c in past_cov_cols if c in group.columns]
            entry["past_covariates"] = {
                col: group[col].fillna(0).tolist() for col in valid_cols
            }
        
        # 未来协变量
        if future_cov_cols:
            last_date = group[date_col].max()
            future_dates = pd.date_range(last_date + timedelta(days=1), 
                                         periods=prediction_length, freq=freq)
            future_df = pd.DataFrame({date_col: future_dates})
            future_df = preprocess_data(
                future_df.assign(asin=item_id, sales_quantity=0), 
                marketplace=marketplace
            )
            valid_cols = [c for c in future_cov_cols if c in future_df.columns]
            entry["future_covariates"] = {
                col: future_df[col].fillna(0).tolist() for col in valid_cols
            }
        
        inputs.append(entry)
    
    return {
        "inputs": inputs,
        "parameters": {
            "prediction_length": prediction_length,
            "freq": freq,
            "quantile_levels": [0.1, 0.5, 0.9]
        }
    }

In [ ]:
payload = build_chronos_payload(
    df,
    prediction_length=PREDICTION_LENGTH,
    freq=FREQ,
    past_cov_cols=PAST_COVARIATES,
    future_cov_cols=FUTURE_COVARIATES,
    marketplace=MARKETPLACE,
)
print(f"预测 {len(payload['inputs'])} 个活跃产品, 每个预测 {PREDICTION_LENGTH} 天")

## 8. 执行预测

In [ ]:
response = predictor.predict(payload)

def response_to_df(response, freq="D"):
    """将响应转换为 DataFrame"""
    dfs = []
    for pred in response["predictions"]:
        forecast_df = pd.DataFrame({
            "asin": pred.get("item_id"),
            "date": pd.date_range(pred["start"], periods=len(pred["mean"]), freq=freq),
            "forecast": pred["mean"],
            "lower_10": pred["0.1"],
            "upper_90": pred["0.9"],
        })
        dfs.append(forecast_df)
    return pd.concat(dfs, ignore_index=True)

forecast_df = response_to_df(response, freq=FREQ)
print(f"预测结果: {len(forecast_df):,} 行")
forecast_df.head()

## 9. 可视化预测结果

In [ ]:
def plot_forecast(df, forecast_df, asin, history_days=90):
    """绘制单个产品的预测图"""
    hist = df[df["asin"] == asin].tail(history_days).set_index("date")["sales_quantity"]
    pred = forecast_df[forecast_df["asin"] == asin].set_index("date")
    
    if len(pred) == 0:
        print(f"产品 {asin} 无预测数据")
        return
    
    plt.figure(figsize=(14, 5))
    plt.plot(hist.index, hist.values, label="历史销量", color="#2E86AB", linewidth=2)
    plt.plot(pred.index, pred["forecast"], label="预测销量", color="#E94F37", linewidth=2)
    plt.fill_between(pred.index, pred["lower_10"], pred["upper_90"], 
                     alpha=0.3, color="#E94F37", label="90% 置信区间")
    
    plt.title(f"产品 {asin} 销量预测", fontsize=14)
    plt.xlabel("日期")
    plt.ylabel("销量")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 绘制前5个产品
for asin in forecast_df["asin"].unique()[:5]:
    plot_forecast(df, forecast_df, asin)

## 10. 保存预测结果到 S3

In [ ]:
# 保存到本地
output_file = f"forecast_{MARKETPLACE}_{datetime.now().strftime('%Y%m%d')}.csv"
forecast_df.to_csv(output_file, index=False)

# 上传到 S3
s3_output_key = f"{S3_PREFIX}/output/{output_file}"
s3.upload_file(output_file, BUCKET, s3_output_key)

print(f"预测结果已保存:")
print(f"  本地: {output_file}")
print(f"  S3: s3://{BUCKET}/{s3_output_key}")

## 11. 清理资源

In [ ]:
# 删除端点 (停止计费)
# predictor.delete_predictor()